# 02. Tiền xử lý dữ liệu & Trích xuất đặc trưng

**Project:** `social-anomaly-clustering`  
**Đề tài:** Ứng dụng kỹ thuật gom nhóm trong phát hiện hành vi bất thường của người dùng mạng xã hội  
**Dataset:** `bot_detection_data.csv`

Notebook này thực hiện bước **tiền xử lý dữ liệu** và **tạo đặc trưng hành vi người dùng** để chuẩn bị cho các thuật toán gom nhóm như **K-Means** và **DBSCAN**.

## Mục tiêu

Từ dữ liệu gốc:

```text
datasets/raw/bot_detection_data.csv
```

Tạo ra các file sau:

```text
datasets/cleaned/bot_detection_clean.csv
datasets/processed/user_features.csv
datasets/processed/user_features_with_label.csv
datasets/processed/user_features_scaled.csv
results/models/scaler.pkl
```

> Lưu ý: Cột `bot_label` chỉ dùng để đánh giá sau clustering, không đưa vào quá trình huấn luyện/gom nhóm.

In [23]:
pip install pandas numpy seaborn matplotlib pyyaml scikit-learn

  Using cached scikit_learn-1.8.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp312-cp312-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp312-cp312-macosx_12_0_arm64.whl (8.1 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.17.1-cp312-cp312-macosx_14_0_arm64.whl (20.3 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Import thư viện và thiết lập đường dẫn

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

DATA_PATH = Path("../datasets/raw/bot_detection_data.csv")
FIGURE_DIR = Path("../results/figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset path:", DATA_PATH)
print("Figure output path:", FIGURE_DIR)

Dataset path: ../datasets/raw/bot_detection_data.csv
Figure output path: ../results/figures


## 2. Đọc dữ liệu gốc

Ở bước này, dữ liệu được đọc từ file gốc trong thư mục `datasets/raw/`.

Ta tạo biến `df_raw` để lưu dữ liệu gốc ban đầu. Sau đó tạo bản sao `df_clean = df_raw.copy()` để thực hiện các bước xử lý. Cách làm này giúp giữ lại dữ liệu gốc để đối chiếu khi cần.

In [4]:
df_raw = pd.read_csv(DATA_PATH)

df_raw.head()

,User ID,Username,Tweet,Retweet Count,Mention Count,Follower Count,Verified,Bot Label,Location,Created At,Hashtags
0,132131,flong,Station activity person against natural majority none few size expect six marriage.,85,1,2353,False,1,Adkinston,2020-05-11 15:29:50,NaN
1,289683,hinesstephanie,Authority research natural life material staff rate common protect attention.,55,5,9617,True,0,Sanderston,2022-11-26 05:18:10,both live
2,779715,roberttran,Manage whose quickly especially foot none to goal range case.,6,2,4363,True,0,Harrisonfurt,2022-08-08 03:16:54,phone ahead
3,696168,pmason,Just cover eight opportunity strong policy which.,54,5,2242,True,1,Martinezberg,2021-08-14 22:27:05,ever quickly new I
4,704441,noah87,Animal sign six data good or.,26,3,8438,False,1,Camachoville,2020-04-13 21:24:21,foreign mention


## 3. Xử lí đổi tên cột

Đổi tên các cột về dạng:

```text
chữ thường + dấu gạch dưới
```

Ví dụ:

```text
User ID        -> user_id
Retweet Count -> retweet_count
Bot Label     -> bot_label
Created At    -> created_at
```


In [5]:
df_raw = df_raw.rename(columns={
    'User ID': 'user_id',
    'Username': 'username',
    'Tweet': 'tweet',
    'Retweet Count': 'retweet_count',
    'Mention Count': 'mention_count',
    'Follower Count': 'follower_count',
    'Verified': 'verified',
    'Bot Label': 'bot_label',
    'Location': 'location',
    'Created At': 'created_at',
    'Hashtags': 'hashtags'
})

# Kiểm tra kết quả
df_raw.head()

,user_id,username,tweet,retweet_count,mention_count,follower_count,verified,bot_label,location,created_at,hashtags
0,132131,flong,Station activity person against natural majority none few size expect six marriage.,85,1,2353,False,1,Adkinston,2020-05-11 15:29:50,NaN
1,289683,hinesstephanie,Authority research natural life material staff rate common protect attention.,55,5,9617,True,0,Sanderston,2022-11-26 05:18:10,both live
2,779715,roberttran,Manage whose quickly especially foot none to goal range case.,6,2,4363,True,0,Harrisonfurt,2022-08-08 03:16:54,phone ahead
3,696168,pmason,Just cover eight opportunity strong policy which.,54,5,2242,True,1,Martinezberg,2021-08-14 22:27:05,ever quickly new I
4,704441,noah87,Animal sign six data good or.,26,3,8438,False,1,Camachoville,2020-04-13 21:24:21,foreign mention


## 4. Xử lý giá trị thiếu

Trong bước khám phá dữ liệu, cột `hashtags` có một số giá trị thiếu.

Trong ngữ cảnh Twitter/X, giá trị thiếu ở `hashtags` có thể hiểu là tweet không sử dụng hashtag. Vì vậy, ta thay giá trị thiếu bằng chuỗi rỗng thay vì xóa dòng.

In [6]:
df_raw["hashtags"] = df_raw["hashtags"].fillna("")

In [7]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   user_id         50000 non-null  int64
 1   username        50000 non-null  str  
 2   tweet           50000 non-null  str  
 3   retweet_count   50000 non-null  int64
 4   mention_count   50000 non-null  int64
 5   follower_count  50000 non-null  int64
 6   verified        50000 non-null  bool 
 7   bot_label       50000 non-null  int64
 8   location        50000 non-null  str  
 9   created_at      50000 non-null  str  
 10  hashtags        50000 non-null  str  
dtypes: bool(1), int64(5), str(5)
memory usage: 3.9 MB


**Nhận xét:**  

Sau khi xử lý, cột `hashtags` không còn giá trị thiếu. Cách xử lý này phù hợp vì không làm mất dữ liệu và vẫn giữ được ý nghĩa: chuỗi rỗng tương ứng với không có hashtag.

## 5. Chuyển đổi kiểu dữ liệu

Một số cột cần được chuyển kiểu trước khi tạo đặc trưng:

| Cột | Kiểu ban đầu | Kiểu sau xử lý | Lý do |
|---|---|---|---|
| `verified` | bool | int | Mô hình clustering cần dữ liệu số |
| `created_at` | object/string | datetime | Dùng để tạo `account_age_days` |

Quy ước chuyển `verified`:

```text
False -> 0
True  -> 1
```

In [8]:
df_raw['verified'] = df_raw['verified'].astype(int)

df_raw['created_at'] = pd.to_datetime(df_raw["created_at"], errors="coerce")

In [27]:
print("Số giá trị created_at bị lỗi sau khi chuyển datetime:", df_clean["created_at"].isna().sum())
print("Ngày nhỏ nhất:", df_clean["created_at"].min())
print("Ngày lớn nhất:", df_clean["created_at"].max())

Số giá trị created_at bị lỗi sau khi chuyển datetime: 0
Ngày nhỏ nhất: 2020-01-01 00:44:14
Ngày lớn nhất: 2023-05-31 07:53:27


**Nhận xét:**  

Cột `verified` đã được chuyển sang dạng số và cột `created_at` đã được chuyển sang dạng thời gian. Nếu số giá trị lỗi của `created_at` bằng 0

In [10]:
df_raw["created_at"].isna().sum()

np.int64(0)

In [11]:
df_clean = df_raw.copy()

df_clean.head()

,user_id,username,tweet,retweet_count,mention_count,follower_count,verified,bot_label,location,created_at,hashtags
0,132131,flong,Station activity person against natural majority none few size expect six marriage.,85,1,2353,0,1,Adkinston,2020-05-11 15:29:50,
1,289683,hinesstephanie,Authority research natural life material staff rate common protect attention.,55,5,9617,1,0,Sanderston,2022-11-26 05:18:10,both live
2,779715,roberttran,Manage whose quickly especially foot none to goal range case.,6,2,4363,1,0,Harrisonfurt,2022-08-08 03:16:54,phone ahead
3,696168,pmason,Just cover eight opportunity strong policy which.,54,5,2242,1,1,Martinezberg,2021-08-14 22:27:05,ever quickly new I
4,704441,noah87,Animal sign six data good or.,26,3,8438,0,1,Camachoville,2020-04-13 21:24:21,foreign mention


In [12]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   user_id         50000 non-null  int64         
 1   username        50000 non-null  str           
 2   tweet           50000 non-null  str           
 3   retweet_count   50000 non-null  int64         
 4   mention_count   50000 non-null  int64         
 5   follower_count  50000 non-null  int64         
 6   verified        50000 non-null  int64         
 7   bot_label       50000 non-null  int64         
 8   location        50000 non-null  str           
 9   created_at      50000 non-null  datetime64[us]
 10  hashtags        50000 non-null  str           
dtypes: datetime64[us](1), int64(6), str(4)
memory usage: 4.2 MB


## 6. Trích xuất đặc trưng mới

Các thuật toán gom nhóm như K-Means và DBSCAN không xử lý trực tiếp tốt các cột văn bản như `tweet`, `username`, `hashtags`. Vì vậy, ta chuyển một số thông tin văn bản/thời gian thành đặc trưng số.

Các đặc trưng mới gồm:

| Feature | Nguồn gốc | Ý nghĩa |
|---|---|---|
| `tweet_length` | `tweet` | Độ dài nội dung tweet |
| `username_length` | `username` | Độ dài tên tài khoản |
| `hashtag_count` | `hashtags` | Số lượng hashtag |
| `has_hashtag` | `hashtags` | Có hashtag hay không |
| `account_age_days` | `created_at` | Số ngày tính từ thời điểm tạo đến ngày mới nhất trong dataset |

In [13]:
# Chọn ngày mới nhất trong dataset làm mốc thời gian
reference_date = df_clean["created_at"].max()

# Độ dài nội dung tweet
df_clean["tweet_length"] = df_clean["tweet"].astype(str).str.len()

# Độ dài username
df_clean["username_length"] = df_clean["username"].astype(str).str.len()

# Số lượng hashtag
df_clean["hashtag_count"] = df_clean["hashtags"].apply(
    lambda x: 0 if str(x).strip() == "" else len(str(x).split())
)

# Có sử dụng hashtag hay không
df_clean["has_hashtag"] = (df_clean["hashtag_count"] > 0).astype(int)

# Số ngày tính từ created_at đến ngày mới nhất trong dataset
df_clean["account_age_days"] = (
    reference_date - df_clean["created_at"]
).dt.days

### ý nghĩa

- `tweet_length`: giúp mô hình nhận biết sự khác biệt về độ dài nội dung tweet.
- `username_length`: có thể phản ánh một số mẫu username bất thường.
- `hashtag_count`: bot/spam account có thể có xu hướng dùng nhiều hashtag để tăng khả năng lan truyền.
- `has_hashtag`: biểu diễn nhị phân việc có dùng hashtag hay không.
- `account_age_days`: biểu diễn độ lâu của bản ghi/tài khoản trong dataset.

Các đặc trưng này giúp chuyển dữ liệu phi cấu trúc hoặc bán cấu trúc thành dạng số để phục vụ gom nhóm.

In [15]:
df_clean[
    [
        "user_id",
        "tweet_length",
        "username_length",
        "hashtag_count",
        "has_hashtag",
        "account_age_days"
    ]
].head()

,user_id,tweet_length,username_length,hashtag_count,has_hashtag,account_age_days
0,132131,83,5,0,0,1114
1,289683,77,14,2,1,186
2,779715,61,10,2,1,296
3,696168,49,6,4,1,654
4,704441,29,6,2,1,1142


In [16]:
df_clean[
    [
        "tweet_length",
        "username_length",
        "hashtag_count",
        "has_hashtag",
        "account_age_days"
    ]
].describe()

,tweet_length,username_length,hashtag_count,has_hashtag,account_age_days
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,62.627340,9.803780,2.500260,0.833180,623.570160
std,16.471543,2.928646,1.709368,0.372819,360.836316
min,23.000000,3.000000,0.000000,0.000000,0.000000
25%,49.000000,7.000000,1.000000,1.000000,312.000000
50%,62.000000,9.000000,2.000000,1.000000,621.000000
75%,76.000000,12.000000,4.000000,1.000000,938.000000
max,118.000000,21.000000,5.000000,1.000000,1246.000000


## 7. Kiểm tra đặc trưng `has_location`

Trong bước EDA, cột `location` gần như không bị thiếu. Ta kiểm tra thử nếu tạo `has_location` thì có tạo ra thông tin phân biệt hay không.

In [28]:
df_clean["has_location"] = df_clean["location"].apply(
    lambda x: 0 if pd.isna(x) or str(x).strip() == "" else 1
)

df_clean["has_location"].value_counts()

has_location
1    50000
Name: count, dtype: int64

**Nhận xét:**  

Nếu `has_location` chỉ có một giá trị duy nhất, ví dụ toàn bộ đều bằng 1, thì đặc trưng này không giúp phân biệt các người dùng. Vì vậy, trong tập đặc trưng chính thức đưa vào clustering, ta **không sử dụng `has_location`**.

## 8. Lưu dữ liệu đã làm sạch

Sau khi đổi tên cột, xử lý thiếu, chuyển kiểu dữ liệu và tạo các feature mới, lưu toàn bộ DataFrame đã làm sạch vào:

```text
datasets/cleaned/bot_detection_clean.csv
```

File này vẫn giữ cả các cột gốc và các cột feature mới để tiện kiểm tra lại.

In [17]:
from pathlib import Path

CLEANED_PATH = Path("../datasets/cleaned/bot_detection_clean.csv")
CLEANED_PATH.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(CLEANED_PATH, index=False)

print("Saved cleaned data to:", CLEANED_PATH)

Saved cleaned data to: ../datasets/cleaned/bot_detection_clean.csv


## 9. Chọn tập đặc trưng dùng cho clustering

Tập đặc trưng chính thức gồm các cột số mô tả hành vi người dùng.

Gồm 4 feature gốc:

```text
retweet_count
mention_count
follower_count
verified
```

và 5 feature mới:

```text
tweet_length
username_length
hashtag_count
has_hashtag
account_age_days
```

Cột `bot_label` **không được dùng làm đầu vào mô hình**, vì đây là nhãn thật. Nhãn này chỉ dùng để đánh giá sau clustering.

In [29]:
feature_cols = [
    "retweet_count",
    "mention_count",
    "follower_count",
    "verified",
    "tweet_length",
    "username_length",
    "hashtag_count",
    "has_hashtag",
    "account_age_days"
]

id_col = "user_id"
label_col = "bot_label"

df_features = df_clean[feature_cols].copy()
df_features_with_label = df_clean[[id_col] + feature_cols + [label_col]].copy()

print("Shape df_features:", df_features.shape)
print("Shape df_features_with_label:", df_features_with_label.shape)

df_features.head()

Shape df_features: (50000, 9)
Shape df_features_with_label: (50000, 11)


,retweet_count,mention_count,follower_count,verified,tweet_length,username_length,hashtag_count,has_hashtag,account_age_days
0,85,1,2353,0,83,5,0,0,1114
1,55,5,9617,1,77,14,2,1,186
2,6,2,4363,1,61,10,2,1,296
3,54,5,2242,1,49,6,4,1,654
4,26,3,8438,0,29,6,2,1,1142


In [30]:
df_features_with_label.head()

,user_id,retweet_count,mention_count,follower_count,verified,tweet_length,username_length,hashtag_count,has_hashtag,account_age_days,bot_label
0,132131,85,1,2353,0,83,5,0,0,1114,1
1,289683,55,5,9617,1,77,14,2,1,186,0
2,779715,6,2,4363,1,61,10,2,1,296,0
3,696168,54,5,2242,1,49,6,4,1,654,1
4,704441,26,3,8438,0,29,6,2,1,1142,1


**Giải thích các file feature:**

| File | Nội dung | Mục đích |
|---|---|---|
| `user_features.csv` | Chỉ gồm các feature số | Đầu vào cho clustering |
| `user_features_with_label.csv` | Gồm `user_id`, feature và `bot_label` | Dùng để đối chiếu và đánh giá sau clustering |

Trong quá trình chạy K-Means/DBSCAN, chỉ sử dụng `user_features.csv` hoặc phiên bản đã chuẩn hóa của nó.

## 10. Kiểm tra dữ liệu feature trước khi chuẩn hóa

Trước khi chuẩn hóa, kiểm tra missing value, kiểu dữ liệu và thống kê mô tả của tập feature.

In [31]:
df_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   retweet_count     50000 non-null  int64
 1   mention_count     50000 non-null  int64
 2   follower_count    50000 non-null  int64
 3   verified          50000 non-null  int64
 4   tweet_length      50000 non-null  int64
 5   username_length   50000 non-null  int64
 6   hashtag_count     50000 non-null  int64
 7   has_hashtag       50000 non-null  int64
 8   account_age_days  50000 non-null  int64
dtypes: int64(9)
memory usage: 3.4 MB


In [32]:
pd.DataFrame({
    "missing_count": df_features.isnull().sum(),
    "missing_percent": (df_features.isnull().sum() / len(df_features) * 100).round(2)
})

,missing_count,missing_percent
retweet_count,0,0.0
mention_count,0,0.0
follower_count,0,0.0
verified,0,0.0
tweet_length,0,0.0
username_length,0,0.0
hashtag_count,0,0.0
has_hashtag,0,0.0
account_age_days,0,0.0


In [33]:
df_features.describe().T

,count,mean,std,min,25%,50%,75%,max
retweet_count,50000.0,50.00560,29.181160,0.0,25.00,50.0,75.0,100.0
mention_count,50000.0,2.51376,1.708563,0.0,1.00,3.0,4.0,5.0
follower_count,50000.0,4988.60238,2878.742898,0.0,2487.75,4991.5,7471.0,10000.0
verified,50000.0,0.50008,0.500005,0.0,0.00,1.0,1.0,1.0
tweet_length,50000.0,62.62734,16.471543,23.0,49.00,62.0,76.0,118.0
username_length,50000.0,9.80378,2.928646,3.0,7.00,9.0,12.0,21.0
hashtag_count,50000.0,2.50026,1.709368,0.0,1.00,2.0,4.0,5.0
has_hashtag,50000.0,0.83318,0.372819,0.0,1.00,1.0,1.0,1.0
account_age_days,50000.0,623.57016,360.836316,0.0,312.00,621.0,938.0,1246.0


**Nhận xét:**  

Các feature có thang đo rất khác nhau. Ví dụ:

```text
follower_count     có thể lên đến hàng chục nghìn
mention_count      chỉ từ 0 đến 5
verified           chỉ gồm 0 và 1
account_age_days   tính theo ngày
```

Do K-Means và DBSCAN đều dựa trên khoảng cách giữa các điểm dữ liệu, cần chuẩn hóa dữ liệu trước khi đưa vào mô hình.

## 11. Lưu tập đặc trưng chưa chuẩn hóa

Lưu hai file:

```text
datasets/processed/user_features.csv
datasets/processed/user_features_with_label.csv
```

In [20]:
from pathlib import Path

PROCESSED_DIR = Path("../datasets/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_PATH = PROCESSED_DIR / "user_features.csv"
FEATURES_WITH_LABEL_PATH = PROCESSED_DIR / "user_features_with_label.csv"

df_features.to_csv(FEATURES_PATH, index=False)
df_features_with_label.to_csv(FEATURES_WITH_LABEL_PATH, index=False)

print("Saved:", FEATURES_PATH)
print("Saved:", FEATURES_WITH_LABEL_PATH)

Saved: ../datasets/processed/user_features.csv
Saved: ../datasets/processed/user_features_with_label.csv


## 12. Chuẩn hóa dữ liệu bằng StandardScaler

Vì các thuật toán clustering như K-Means và DBSCAN dựa vào khoảng cách, các feature cần được đưa về cùng thang đo.

`StandardScaler` chuyển mỗi feature về dạng có:

```text
mean ≈ 0
standard deviation ≈ 1
```

Công thức chuẩn hóa:

```text
z = (x - mean) / standard_deviation
```

Sau chuẩn hóa, các feature như `follower_count`, `mention_count`, `verified`, `hashtag_count` sẽ có mức ảnh hưởng cân bằng hơn trong quá trình tính khoảng cách.

In [24]:
from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()

scaled_features = scaler.fit_transform(df_features)

df_features_scaled = pd.DataFrame(
    scaled_features,
    columns=feature_cols
)

df_features_scaled.head()

,retweet_count,mention_count,follower_count,verified,tweet_length,username_length,hashtag_count,has_hashtag,account_age_days
0,1.199224,-0.885993,-0.915548,-1.00016,1.236852,-1.640290,-1.462696,-2.234834,1.359161
1,0.171153,1.455179,1.607800,0.99984,0.872584,1.432833,-0.292661,0.447460,-1.212668
2,-1.508029,-0.300700,-0.217320,0.99984,-0.098798,0.067001,-0.292661,0.447460,-0.907817
3,0.136884,1.455179,-0.954107,0.99984,-0.827335,-1.298831,0.877374,0.447460,0.084332
4,-0.822649,0.284593,1.198243,-1.00016,-2.041562,-1.298831,-0.292661,0.447460,1.436760


In [34]:
scaled_check = pd.DataFrame({
    "mean_after_scaling": df_features_scaled.mean().round(6),
    "std_after_scaling": df_features_scaled.std().round(6)
})

scaled_check

,mean_after_scaling,std_after_scaling
retweet_count,-0.0,1.00001
mention_count,0.0,1.00001
follower_count,-0.0,1.00001
verified,0.0,1.00001
tweet_length,0.0,1.00001
username_length,0.0,1.00001
hashtag_count,0.0,1.00001
has_hashtag,-0.0,1.00001
account_age_days,0.0,1.00001


In [25]:
df_features_scaled.describe().T

,count,mean,std,min,25%,50%,75%,max
retweet_count,50000.0,-4.177991e-17,1.00001,-1.713643,-0.856918,-0.000192,0.856534,1.713259
mention_count,50000.0,1.435296e-17,1.00001,-1.471286,-0.885993,0.284593,0.869886,1.455179
follower_count,50000.0,-9.293899e-17,1.00001,-1.732927,-0.868739,0.001007,0.862329,1.740846
verified,50000.0,9.379164e-18,1.00001,-1.000160,-1.000160,0.999840,0.999840,0.999840
tweet_length,50000.0,2.023626e-16,1.00001,-2.405830,-0.827335,-0.038087,0.811873,3.361750
username_length,50000.0,9.208634e-17,1.00001,-2.323206,-0.957373,-0.274457,0.749917,3.823040
hashtag_count,50000.0,1.875833e-17,1.00001,-1.462696,-0.877678,-0.292661,0.877374,1.462392
has_hashtag,50000.0,-9.833911e-17,1.00001,-2.234834,0.447460,0.447460,0.447460,0.447460
account_age_days,50000.0,4.092726e-17,1.00001,-1.728142,-0.863475,-0.007123,0.871401,1.724982


**Nhận xét:**  

Các giá trị sau chuẩn hóa có thể âm hoặc dương. Đây là điều bình thường.

Ví dụ:

```text
-1.2  -> thấp hơn trung bình khoảng 1.2 độ lệch chuẩn
0     -> gần mức trung bình
1.5   -> cao hơn trung bình khoảng 1.5 độ lệch chuẩn
```

Dữ liệu sau chuẩn hóa không còn giữ đơn vị gốc, nhưng phù hợp hơn để đưa vào các thuật toán dựa trên khoảng cách.

## 13. Lưu dữ liệu đã chuẩn hóa và scaler

Lưu dữ liệu đã chuẩn hóa vào:

```text
datasets/processed/user_features_scaled.csv
```

Lưu scaler vào:

```text
results/models/scaler.pkl
```

File `scaler.pkl` giúp tái sử dụng lại cùng cách chuẩn hóa nếu cần xử lý dữ liệu mới hoặc chạy lại mô hình.

In [26]:
from pathlib import Path
import joblib

PROCESSED_DIR = Path("../datasets/processed")
MODEL_DIR = Path("../results/models")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SCALED_FEATURES_PATH = PROCESSED_DIR / "user_features_scaled.csv"
SCALER_PATH = MODEL_DIR / "scaler.pkl"

df_features_scaled.to_csv(SCALED_FEATURES_PATH, index=False)
joblib.dump(scaler, SCALER_PATH)

print("Saved scaled features to:", SCALED_FEATURES_PATH)
print("Saved scaler to:", SCALER_PATH)

Saved scaled features to: ../datasets/processed/user_features_scaled.csv
Saved scaler to: ../results/models/scaler.pkl
